# 02 — Customer Intelligence: 10 Business SQL Queries

**Dataset:** 130,038 real Amazon Electronics reviews (5-core subset)
**Engine:** SQLite in-memory (loaded from CSV)
**Source:** http://jmcauley.ucsd.edu/data/amazon/

In [ ]:
import pandas as pd
import sqlite3
import numpy as np

# Load into SQLite
df = pd.read_csv('../data/amazon_reviews_electronics_5core.csv')
df['reviewDate'] = pd.to_datetime(df['unixReviewTime'], unit='s')
df['reviewYear'] = df['reviewDate'].dt.year
df['reviewMonth'] = df['reviewDate'].dt.month
df['reviewLength'] = df['reviewText'].fillna('').str.len()

conn = sqlite3.connect(':memory:')
df.to_sql('reviews', conn, index=False, if_exists='replace')

print(f'Loaded {len(df):,} rows into SQLite')
print(f'Date range: {df.reviewDate.min().date()} to {df.reviewDate.max().date()}')

## Q1 — Product Performance Ranking

Rank products by avg rating, review volume, and helpfulness (minimum 10 reviews).

In [ ]:
query1 = """
SELECT 
    asin AS product_id,
    COUNT(*) AS review_volume,
    ROUND(AVG(overall), 2) AS avg_rating,
    ROUND(AVG(CASE WHEN helpful_total > 0 THEN 1.0*helpful_upvotes/helpful_total END), 2) AS avg_helpfulness,
    SUM(helpful_total) AS total_votes
FROM reviews
GROUP BY asin
HAVING review_volume >= 10
ORDER BY avg_rating DESC, review_volume DESC
LIMIT 15
"""
pd.read_sql(query1, conn)

## Q2 — Rating Degradation Over Time

Are products losing trust? Track average rating by product age cohort.

In [ ]:
query2 = """
WITH product_first_review AS (
    SELECT asin, MIN(reviewDate) AS first_review_date
    FROM reviews
    GROUP BY asin
),
product_age AS (
    SELECT 
        r.asin,
        r.overall,
        r.reviewDate,
        p.first_review_date,
        CAST((julianday(r.reviewDate) - julianday(p.first_review_date)) / 30 AS INTEGER) AS months_since_first
    FROM reviews r
    JOIN product_first_review p ON r.asin = p.asin
)
SELECT 
    months_since_first AS product_age_months,
    COUNT(*) AS review_count,
    ROUND(AVG(overall), 2) AS avg_rating
FROM product_age
WHERE months_since_first BETWEEN 0 AND 24
GROUP BY months_since_first
ORDER BY months_since_first
"""
pd.read_sql(query2, conn)

## Q3 — Review Velocity Analysis

Reviews per month, aggregated.

In [ ]:
query3 = """
SELECT 
    CAST(strftime('%Y', reviewDate) AS INTEGER) AS year,
    CAST(strftime('%m', reviewDate) AS INTEGER) AS month,
    COUNT(*) AS reviews_per_month,
    ROUND(AVG(overall), 2) AS avg_rating
FROM reviews
GROUP BY year, month
ORDER BY year, month
"""
velocity = pd.read_sql(query3, conn)
print(f'Months covered: {len(velocity)}')
print(f'Avg monthly volume: {velocity.reviews_per_month.mean():.0f}')
print(f'Peak month volume: {velocity.reviews_per_month.max():,}')
velocity.tail(12)

## Q4 — Helpfulness Scoring: What Makes a Review Helpful?

In [ ]:
query4 = """
SELECT 
    CASE 
        WHEN reviewLength < 100 THEN 'Short (<100)'
        WHEN reviewLength < 500 THEN 'Medium (100-500)'
        ELSE 'Long (500+)'
    END AS length_tier,
    COUNT(*) AS review_count,
    ROUND(AVG(overall), 2) AS avg_rating,
    ROUND(AVG(CASE WHEN helpful_total > 0 THEN 1.0*helpful_upvotes/helpful_total END), 2) AS avg_helpfulness,
    ROUND(AVG(helpful_total), 1) AS avg_votes_received
FROM reviews
GROUP BY length_tier
ORDER BY avg_helpfulness DESC
"""
pd.read_sql(query4, conn)

## Q5 — Sentiment Shift Detection (Year-over-Year Rating Changes)

In [ ]:
query5 = """
SELECT 
    CAST(strftime('%Y', reviewDate) AS INTEGER) AS year,
    COUNT(*) AS review_count,
    ROUND(AVG(overall), 2) AS avg_rating,
    ROUND(AVG(CASE WHEN helpful_total > 0 THEN 1.0*helpful_upvotes/helpful_total END), 2) AS avg_helpfulness,
    ROUND(AVG(reviewLength), 0) AS avg_review_length
FROM reviews
WHERE CAST(strftime('%Y', reviewDate) AS INTEGER) BETWEEN 2005 AND 2014
GROUP BY year
ORDER BY year
"""
pd.read_sql(query5, conn)

## Q6 — Product Lifecycle: Early Reviews vs. Mature Reviews

In [ ]:
query6 = """
WITH ranked_reviews AS (
    SELECT 
        asin,
        overall,
        reviewDate,
        ROW_NUMBER() OVER (PARTITION BY asin ORDER BY reviewDate) AS review_seq,
        COUNT(*) OVER (PARTITION BY asin) AS total_reviews
    FROM reviews
)
SELECT 
    CASE 
        WHEN review_seq <= 3 THEN 'Early (1st-3rd)'
        WHEN review_seq <= 10 THEN 'Growth (4th-10th)'
        ELSE 'Mature (11th+)'
    END AS lifecycle_stage,
    COUNT(*) AS review_count,
    ROUND(AVG(overall), 2) AS avg_rating,
    ROUND(AVG(CASE WHEN helpful_total > 0 THEN 1.0*helpful_upvotes/helpful_total END), 2) AS avg_helpfulness
FROM ranked_reviews
WHERE total_reviews >= 5
GROUP BY lifecycle_stage
ORDER BY 
    CASE lifecycle_stage
        WHEN 'Early (1st-3rd)' THEN 1
        WHEN 'Growth (4th-10th)' THEN 2
        ELSE 3
    END
"""
pd.read_sql(query6, conn)

## Q7 — Review Length Correlation with Helpfulness and Rating

In [ ]:
query7 = """
SELECT 
    ROUND(overall) AS star_rating,
    ROUND(AVG(reviewLength), 0) AS avg_length,
    ROUND(AVG(CASE WHEN helpful_total > 0 THEN 1.0*helpful_upvotes/helpful_total END), 2) AS avg_helpfulness,
    ROUND(AVG(helpful_total), 1) AS avg_total_votes
FROM reviews
WHERE helpful_total > 0
GROUP BY ROUND(overall)
ORDER BY star_rating
"""
pd.read_sql(query7, conn)

## Q8 — Seasonal Patterns in Review Volume

Black Friday (Nov), Prime Day (July proxy), Holiday (Dec).

In [ ]:
query8 = """
SELECT 
    CAST(strftime('%m', reviewDate) AS INTEGER) AS month,
    COUNT(*) AS review_count,
    ROUND(AVG(overall), 2) AS avg_rating
FROM reviews
GROUP BY month
ORDER BY month
"""
seasonal = pd.read_sql(query8, conn)
print(seasonal.to_string(index=False))

## Q9 — Customer Engagement Tiers (Power Reviewers vs. Casual)

In [ ]:
query9 = """
SELECT 
    CASE 
        WHEN review_count >= 50 THEN 'Power (50+)'
        WHEN review_count >= 10 THEN 'Active (10-49)'
        WHEN review_count >= 3 THEN 'Casual (3-9)'
        ELSE 'One-off (1-2)'
    END AS engagement_tier,
    COUNT(*) AS reviewer_count,
    ROUND(AVG(review_count), 1) AS avg_reviews_per_reviewer,
    ROUND(AVG(avg_rating), 2) AS avg_rating_given,
    ROUND(SUM(review_count) * 100.0 / (SELECT COUNT(*) FROM reviews), 1) AS pct_of_total_reviews
FROM (
    SELECT 
        reviewerID,
        COUNT(*) AS review_count,
        AVG(overall) AS avg_rating
    FROM reviews
    GROUP BY reviewerID
)
GROUP BY engagement_tier
ORDER BY avg_reviews_per_reviewer DESC
"""
pd.read_sql(query9, conn)

## Q10 — Churn Signal: Products with Declining Review Ratings

In [ ]:
query10 = """
WITH product_trend AS (
    SELECT 
        asin,
        CAST(strftime('%Y', reviewDate) AS INTEGER) AS year,
        AVG(overall) AS year_avg_rating,
        COUNT(*) AS year_review_count
    FROM reviews
    GROUP BY asin, year
    HAVING year_review_count >= 5
),
yoy_change AS (
    SELECT 
        a.asin,
        a.year AS current_year,
        a.year_avg_rating AS current_rating,
        b.year_avg_rating AS prior_rating,
        a.year_avg_rating - b.year_avg_rating AS rating_change
    FROM product_trend a
    JOIN product_trend b ON a.asin = b.asin AND a.year = b.year + 1
    WHERE b.year_avg_rating > 0
)
SELECT 
    current_year AS year,
    COUNT(*) AS products_with_data,
    ROUND(AVG(rating_change), 2) AS avg_rating_change,
    COUNT(CASE WHEN rating_change < -0.5 THEN 1 END) AS sharp_decliners,
    COUNT(CASE WHEN rating_change > 0.5 THEN 1 END) AS improvers
FROM yoy_change
GROUP BY current_year
ORDER BY current_year
"""
pd.read_sql(query10, conn)